# arXiv — exploratory data analysis

Title and abstract text of arXiv papers (2000–2025), each labelled with one or more top-level subjects (multi-label classification, scored on macro AUROC). The dataset is used to study temporal drift, so the views below focus on how submission volume, subject mix, and text shift across the years.

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

from drift_happens.dataset.arxiv.load import load_arxiv

plt.rcParams["axes.unicode_minus"] = False
PRIMARY, SECONDARY = "#4C78A8", "#E45756"

arxiv = load_arxiv()
arxiv[["created", "title", "top_subjects", "year"]].head()

## Dataset at a glance

In [ ]:
missing = arxiv[["title_abstract", "top_subjects", "year"]].isna().mean().max()
pd.Series({
    "Samples": f"{len(arxiv):,}",
    "Years": f"{arxiv['year'].min()}–{arxiv['year'].max()}",
    "Top-level subjects": str(arxiv["top_subjects"].explode().nunique()),
    "Missing (text, subjects, year)": f"{missing:.1%}",
    "Task": "multi-label classification",
    "Metric": "macro AUROC",
}, name="arxiv")

## Temporal coverage

In [ ]:
per_year = arxiv["year"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(per_year.index, per_year.values, color=PRIMARY)
ax.set_xlabel("Year")
ax.set_ylabel("Papers")
ax.set_title("arXiv submissions per year")
fig.tight_layout()

## Subjects

In [ ]:
subject_totals = arxiv["top_subjects"].explode().value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
subject_totals.sort_values().plot.barh(ax=ax, color=PRIMARY)
ax.set_xlabel("Papers")
ax.set_title("Papers per top-level subject")
fig.tight_layout()

## Subject mix over time

The drift the benchmark targets: each subject's share of submissions, year by year.

In [ ]:
by_year = (
    arxiv[["year", "top_subjects"]]
    .explode("top_subjects")
    .groupby("year")["top_subjects"]
    .value_counts()
    .unstack(fill_value=0)
)
share = by_year.div(by_year.sum(axis=1), axis=0)
order = share.sum().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(11, 6))
im = ax.imshow(share[order].T.values, aspect="auto", cmap="magma")
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7)
ticks = range(0, len(share.index), 3)
ax.set_xticks(list(ticks))
ax.set_xticklabels([share.index[i] for i in ticks])
ax.set_xlabel("Year")
ax.set_title("Subject share over time")
fig.colorbar(im, ax=ax, label="Share of papers")
fig.tight_layout()

## Labels per paper

In [ ]:
cardinality = arxiv["top_subjects"].str.len().value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cardinality.index, cardinality.values, color=PRIMARY)
ax.set_xlabel("Top-level subjects per paper")
ax.set_ylabel("Papers")
ax.set_title("Multi-label cardinality")
fig.tight_layout()

## Text length

In [ ]:
words = arxiv["title_abstract"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(words.clip(upper=400), bins=50, color=PRIMARY)
axes[0].set_xlabel("Words in title + abstract")
axes[0].set_ylabel("Papers")
axes[0].set_title("Text length")

median_words = words.groupby(arxiv["year"]).median()
axes[1].plot(median_words.index, median_words.values, color=PRIMARY)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Median words")
axes[1].set_title("Median text length over time")
fig.tight_layout()

## Vocabulary drift

Document frequency of each word in the earliest fifth of years versus the latest fifth; the terms that rise are the vocabulary the corpus drifts toward.

In [ ]:
def document_frequency(texts):
    counts = Counter()
    for text in texts:
        counts.update(set(re.findall(r"[a-z]{4,}", text.lower())))
    return counts

low, high = arxiv["year"].quantile(0.2), arxiv["year"].quantile(0.8)
early = arxiv.loc[arxiv["year"] <= low, "title_abstract"]
late = arxiv.loc[arxiv["year"] >= high, "title_abstract"]
early = early.sample(min(40000, len(early)), random_state=0)
late = late.sample(min(40000, len(late)), random_state=0)

terms = pd.DataFrame({
    "early": pd.Series(document_frequency(early)) / len(early),
    "late": pd.Series(document_frequency(late)) / len(late),
}).fillna(0)
rising = (terms["late"] - terms["early"]).sort_values().tail(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(rising.index, rising.values, color=PRIMARY)
ax.set_xlabel("Increase in document frequency (late minus early)")
ax.set_title(f"Rising terms: up to {int(low)} vs from {int(high)}")
fig.tight_layout()

## Sample records

In [ ]:
from IPython.display import Markdown, display

records = []
for row in arxiv.sample(4, random_state=0).itertuples():
    abstract = " ".join(row.abstract.split())[:300]
    records.append(
        f"**{row.title.strip()}** · {row.year} · {', '.join(row.top_subjects)}\n\n"
        f"> {abstract}…"
    )
display(Markdown("\n\n---\n\n".join(records)))